# UrbanSense — Tier 1 Road Damage Detection

This notebook documents the Tier 1 road-damage training workflow used for UrbanSense. It uses YOLO11 with the Unified Road Defect Dataset / RDD2022-derived four-class road-damage taxonomy. The original working notebook was executed in Google Colab with a T4 GPU.

Classes: Longitudinal Crack (D00), Transverse Crack (D10), Alligator Crack (D20), Pothole (D40).

Original Colab notebook: https://colab.research.google.com/drive/19fR4fJ5RlkX9h76Iwxh9iC0l2H0Wn2E_


In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics datasets huggingface_hub

import torch
import ultralytics
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))
print('Ultralytics:', ultralytics.__version__)

In [ ]:
from huggingface_hub import hf_hub_download
import os

repo = 'TamAko783/Unified_Road_Defect_Dataset'
dataset_dir = '/content/drive/MyDrive/UrbanSense/dataset'
os.makedirs(dataset_dir, exist_ok=True)

files = ['data/train_a.tar.gz','data/train_b.tar.gz','data/val.tar.gz','rdd_merged.yaml']
for filename in files:
    output_name = os.path.basename(filename)
    destination = os.path.join(dataset_dir, output_name)
    if os.path.exists(destination) and os.path.getsize(destination) > 0:
        print('Already exists:', output_name)
        continue
    downloaded = hf_hub_download(repo_id=repo, filename=filename, repo_type='dataset')
    !cp "{downloaded}" "{destination}"
    print('Saved:', destination)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

urban_root = '/content/drive/.shortcut-targets-by-id/1_kOtmdyII-Hpwi3TVZ_8g0nObETlhrxj/UrbanSense'
dataset_drive = os.path.join(urban_root, 'dataset')
dataset_local = '/content/urban_dataset'
os.makedirs(dataset_local, exist_ok=True)

import tarfile
for archive in ['train_a.tar.gz','train_b.tar.gz','val.tar.gz']:
    archive_path = os.path.join(dataset_drive, archive)
    print('Extracting', archive)
    with tarfile.open(archive_path, 'r:gz') as tar:
        tar.extractall(dataset_local)
print('Extraction complete.')

In [ ]:
import yaml

urban_yaml = {
    'path': '/content/urban_dataset',
    'train': 'images/train',
    'val': 'images/val',
    'nc': 4,
    'names': {
        0: 'Longitudinal Crack (D00)',
        1: 'Transverse Crack (D10)',
        2: 'Alligator Crack (D20)',
        3: 'Pothole (D40)'
    }
}

yaml_path = '/content/urban_dataset/urbansense.yaml'
with open(yaml_path, 'w') as f:
    yaml.safe_dump(urban_yaml, f, sort_keys=False)
print('Created:', yaml_path)

In [ ]:
from glob import glob

train_images = glob('/content/urban_dataset/images/train/*.jpg')
train_labels = glob('/content/urban_dataset/labels/train/*.txt')
val_images = glob('/content/urban_dataset/images/val/*.jpg')
val_labels = glob('/content/urban_dataset/labels/val/*.txt')
print('Training images:', len(train_images))
print('Training labels:', len(train_labels))
print('Validation images:', len(val_images))
print('Validation labels:', len(val_labels))

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
results = model.train(
    data='/content/urban_dataset/urbansense.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project='/content/drive/MyDrive/UrbanSense/checkpoints',
    name='road_damage_v1',
    exist_ok=True,
    save=True,
    save_period=5,
    patience=10,
    workers=2,
    degrees=5,
    translate=0.1,
    scale=0.5,
    fliplr=0.5
)

In [ ]:
# Resume an interrupted training run from the shared UrbanSense checkpoint.
from ultralytics import YOLO
checkpoint = urban_root + '/checkpoints/road_damage_v1/weights/last.pt'
model = YOLO(checkpoint)
results = model.train(resume=True)

In [ ]:
# Inspect the trained checkpoint and class mapping.
from ultralytics import YOLO
model = YOLO(checkpoint)
print(model.names)
print('Starting epoch:', model.ckpt.get('epoch', 'unknown'))

## Training notes

The working run used 25,677 training images and 4,509 validation images. Training was performed on a Google Colab NVIDIA T4 GPU and used YOLO11n. The run was resumable from `last.pt` stored under the shared UrbanSense checkpoints directory.

The repository intentionally does not include the large dataset archives or `.pt` weights; those remain in Google Drive / external dataset storage.
